# Movie Content Based Recommender

In a temporal jupyter notebook I tried 3 approaches:

- TF-IDF + CountVectorizer
- all-mpnet-base-v2: Based on Microsoft MPNet
- all-MiniLM-L6-v2: Based on MiniLM, a lightweight model derived from BERT

The transformers I used were pretrained specifically for sentence similarity tasks, the best one was all-mpnet-base-v2, so in this Jupyter Notebook, I will develop this transformer-based approach, but focusing on the important things, since the trail notebook was very large and it was very confusing  to keep track, so I will keep in this notebook the most important steps to generate the content-based recommender system.

I decided to use transformers since this approach leverage all available textual features, as transformer models are capable of capturing rich semantic information across diverse inputs.

In [ ]:
import torch
import warnings
import numpy as np
import pandas as pd
from tabulate import tabulate
from transformers import logging
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Suppress warnings
logging.set_verbosity_error()
warnings.filterwarnings("ignore")

# Set the maximum display of a column to 120 chars in pandas
pd.set_option('display.max_colwidth', 120)

In [ ]:
mdf = pd.read_csv('movies_final.csv')
mdf.head().transpose()

In [ ]:
# Select the relevant columns
df = mdf[['movieId', 'id', 'title', 'genres', 'overview',
           'tagline', 'cast', 'director', 'keywords']].copy()

print(f'The number of movies is: {df.shape[0]:,}')

In [ ]:
df.isna().sum()

In [ ]:
# Delete the Nan in the tagline column
df['tagline'] = df['tagline'].fillna('')

# Combine the overview with the tagline
df['description'] = df['overview'] + ' ' + df['tagline']

In [ ]:
df['title'] = df['title'].apply(lambda x: f"Title: {x}")
df['director'] = df['director'].apply(lambda x: f"Director: {x}")
df['genres'] = df['genres'].apply(lambda x: f"Genres: {x}")
df['cast'] = df['cast'].apply(lambda x: f"Cast: {x}")
df['keywords'] = df['keywords'].apply(lambda x: f"Keywords: {x}")
df['description'] = df['description'].apply(lambda x: f"Movie Overview: {x}")

In [ ]:
df.head()

## Transformers

These models are capable of capturing deeper semantic relationships between films by understanding the meaning behind the text, rather than relying purely on keyword matching. I will explore three strategies:

The features I will use are: **Title, Director, Genres, Cast, Keywords, Description (Overview + Tagline)**


In [ ]:
def encode_batchwise_structured(model, df, emb_size, batch_size=64):

    # Define the fields to encode
    fields = ['title', 'genres', 'director', 'overview', 'cast', 'keywords']
    num_items = len(df)

    # Preallocate array: (num_fields, num_items, embedding_size)
    all_embeddings = np.empty((len(fields), num_items, emb_size))

    # Encode each field separately
    for idx, field in enumerate(fields):
        field_embeddings = []
        texts = df[field].fillna("").astype(str).tolist()

        for i in tqdm(range(0, num_items, batch_size), desc=f'Encoding {field}'):
            batch = texts[i:i+batch_size]

            with torch.no_grad():
                batch_embeddings = model.encode(
                    batch,
                    batch_size=batch_size,
                    # device=device,
                    normalize_embeddings=True
                )

            field_embeddings.extend(batch_embeddings)

        all_embeddings[idx] = np.array(field_embeddings)

    return all_embeddings

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('all-mpnet-base-v2', device=device)

In [ ]:
embeddings = encode_batchwise_structured(model, df, 768)

## Cosine Similarity

I will need to compute the cosine similarity matrix now, I will use this metric since is the best one for semantic similarity

In [ ]:
def get_structured_cosine_similarity(embeddings, n_items, weights):
    n_features = embeddings.shape[0]  # dimensions: [n_features, n_items, emb_size]
    similarities = np.empty((n_features, n_items, n_items))

    # Compute cosine similarity for each feature
    for i in range(n_features):
        sim_matrix = cosine_similarity(embeddings[i])
        similarities[i] = sim_matrix * weights[i]

    # Return weighted cosine similarity
    return np.sum(similarities, axis=0)

In [ ]:
# Number of items (movies)
n = df.shape[0]

# Create the weights for each feature
# Title, Director, Genres, Cast, Keywords, Description
weights = np.array([0.05, 0.15, 0.2, 0.1, 0.2, 0.3])

In [ ]:
cosine_sim = get_structured_cosine_similarity(embeddings, n, weights)
cosine_sim.shape

In [ ]:
cosine_sim.dtype

In [ ]:
cosine_sim = cosine_sim.astype(np.float32)
np.save("cosine_sim.npy", cosine_sim)

## Recommendation Function

In [ ]:
def recommend_top_ten(title, cosine_sim_array):

  try:
    # Get the index of the movie that matches the title
    idx = mdf[mdf['title'] == title].index[0]
  except:
    print('Movie not found')
    return

  # Compute the pairwise similarity scores for that movie
  sim_scores = list(enumerate(cosine_sim_array[idx]))

  # Sort the movies based on similarity scores (excluding itself)
  sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:11]

  # Get the indices and scores of the top matches
  movie_indices = [i[0] for i in sim_scores]
  scores = [i[1] for i in sim_scores]

  # Return the recommendations and similarity
  titles = mdf['title'].iloc[movie_indices].to_list()
  result = list(zip(titles, scores))

  return result

### Testing the Recommendations

Now that I have built the recommendation function, I will check the recommenations with 3 different kind of Movies

In [ ]:
def model_recommendations(movie_title):
    """
    Compare top 10 recommendations from multiple similarity matrices.
    """

    # Headers for the recommendation table
    headers = ['Moive', 'Similarity']
    # Get top 10 recommendations per similarity matrix
    recommendations = recommend_top_ten(movie_title, cosine_sim)

    # Build a table: 10 rows, one per rank
    table = []
    for i in range(10):  # Top 10
        row = []
        title, score = recommendations[i]
        row.append(f"{title}")
        row.append(f"({score:.2f})")
        table.append(row)

    # Print the comparison table
    print(tabulate(table, headers=headers, tablefmt="fancy_outline"))

In [ ]:
model_recommendations('Harry Potter and the Philosopher\'s Stone')

In [ ]:
model_recommendations('Your Name.')

In [ ]:
model_recommendations('Inception')

In [ ]:
model_recommendations('Fight Club')

- The weighting allowed to give greater importance to more discriminative features such as genre and description, while reducing the influence of features that could introduce noise.
- This approach generated the most accurate recommendations for all test cases, especially notable in distinctive films such as “Your Name” (recommending other works by Makoto Shinkai) and “Fight Club” (suggesting dark psychological films and works by David Fincher).
- Similarities were the highest and most significant, ranging from 0.65-0.85 for highly relevant recommendations.
- MPNet with custom weights proved to be the most effective combination, capturing both thematic and stylistic similarities.

### Personalized Recommendations

Now that have seen that the recommendations are good so far, I will go one step ahead and personalize a little bit more the recommendations, for this I will get the mean average of all the embeddings that the user has seen and then get the similarity of the movies with the average embeddings

In [ ]:
np.mean(cosine_sim[[0, 1]], axis=0).shape

In [ ]:
def get_personalized_recommendations(movies_liked: list, cosine_sim: np.array, k=10):

    scores = np.mean(cosine_sim[movies_liked], axis=0)

    # Exclude already watched movies
    for idx in movies_liked:
        scores[idx] = -np.inf

    top_indices = np.argsort(scores)[-k:][::-1]
    return df.iloc[top_indices]['title']

Lets try this with 3 different approaches to see if it is vaible

In [ ]:
df[df['title'].isin(['Se7en', 'Your Name.', 'Inception'])]

In [ ]:
get_personalized_recommendations([1816, 3194, 4913], cosine_sim)

The recommendations seems to be okay, since the Inception and Se7en are very similar and the recommendations are alike to that movies

In [ ]:
liked_idx = df[df['title'].isin(['Parasite', 'Your Name.', 'The Dark Knight'])].index
get_personalized_recommendations(liked_idx, cosine_sim)